# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелиной Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

Для поднятия настроения — басня Эрдмана:

```
СМЕТАНА.

Мы любим подмечать у недругов изъяны,
И направлять на них насмешки остриё.

Однажды молоко спросило у сметаны:
- Скажите, вы еда или питье?
Сметана молвила:
- Оставьте ваши шутки!
Действительно, я где-то в промежутке,
Но ведь важна не эта сторона
Всего важнее то, что я вкусна
И все, как правило, бывают мною сыты.

Вот так же точно и гермафродиты:
Тот, кто на свет их произвел,
Конечно, допустил ужасную небрежность.
Но ведь в конце-концов
существенен не пол,
А классовая принадлежность.
```

Читать другие басни: https://stihi.ru/diary/valery31/2014-10-11

In [1]:
import regex as re
import pandas as pd
from bs4 import BeautifulSoup
import pymorphy3
from razdel import tokenize

with open('457615.fb2', encoding='utf-8') as file:
    book = file.read()

body = book.split('<body>')[1]
body = body.split('</body>')[0]
soup = BeautifulSoup(body, 'html.parser')

for section in soup.find_all('section'):
    title = section.find('title')
    if title and title.get_text().strip() == '1928-1935 гг.':
        section_text = str(section)
        blocks = section_text.split('<subtitle>')[1:]
        break

In [2]:
months = {
    'январь': 1, 'января': 1, 'янв': 1,
    'февраль': 2, 'февраля': 2, 'февр': 2, 'фев': 2,
    'март': 3, 'марта': 3, 'мар': 3,
    'апрель': 4, 'апреля': 4, 'апр': 4,
    'май': 5, 'мая': 5,
    'июнь': 6, 'июня': 6, 'июн': 6,
    'июль': 7, 'июля': 7, 'июл': 7,
    'август': 8, 'августа': 8, 'авг': 8,
    'сентябрь': 9, 'сентября': 9, 'сент': 9, 'сен': 9,
    'октябрь': 10, 'октября': 10, 'окт': 10,
    'ноябрь': 11, 'ноября': 11, 'нояб': 11, 'ноя': 11,
    'декабрь': 12, 'декабря': 12, 'дек': 12
}

roman_months = {
    'i': 1, 'ii': 2, 'iii': 3, 'iv': 4, 'v': 5, 'vi': 6,
    'vii': 7, 'viii': 8, 'ix': 9, 'x': 10, 'xi': 11, 'xii': 12
}

def read_date(line, current_year):
    line = line.lower().strip('[]() ')
    line = line.replace('.', ' ')
    line = line.replace('/', ' ')
    line = line.replace('-', ' ')
    parts = line.split()

    if parts and parts[-1] in ['г', 'год', 'года']:
        parts = parts[:-1]

    year = current_year
    if parts and parts[-1].isdigit():
        last = parts[-1]
        if len(last) == 4:
            year = int(last)
            parts = parts[:-1]
            if not parts:
                return [str(year), year]
        elif len(last) == 2 and len(parts) > 1:
            year = int('19' + last)
            parts = parts[:-1]

    if len(parts) not in [1, 2]:
        return [None, current_year]

    month = parts[-1]
    if month in months:
        month = months[month]
    elif year and len(parts) == 2 and month in roman_months:
        month = roman_months[month]
    elif year and len(parts) == 2 and month.isdigit():
        month = int(month)
    else:
        return [None, current_year]

    day = None
    if len(parts) == 2:
        if not parts[0].isdigit():
            return [None, current_year]
        day = int(parts[0])
        if day < 1 or day > 31:
            return [None, current_year]
    if month < 1 or month > 12:
        return [None, current_year]
    if year is None:
        return [None, current_year]

    date = str(year) + '-'
    if month < 10:
        date = date + '0'
    date = date + str(month)
    if day is not None:
        date = date + '-'
        if day < 10:
            date = date + '0'
        date = date + str(day)
    return [date, year]

In [3]:
address_lines = [
    'Москва', 'Заказное', 'Начальнику Строительства',
    'Большая Мурта', 'Деревня Большая Мурта'
]
address_starts = [
    'проезд Художественного театра', 'Художественный театр',
    'Ангелине Осиповне Степановой', 'А. О. Степановой',
    'Ольге Алексеевне Ростовцевой', 'Баку,', 'Оперный театр',
    'Батум,', 'Чаква,', 'Енисейск, ул.', 'Енисейск. Сталина',
    'Енисейск, Сталина', 'Томск.'
]

rows = []
current_year = None

for block in blocks:
    block_parts = block.split('</subtitle>')
    heading = block_parts[0]
    block_text = block_parts[1]
    block_soup = BeautifulSoup(block_text, 'html.parser')
    paragraph_tags = block_soup.find_all('p')

    paragraphs = []
    for paragraph in paragraph_tags:
        line = paragraph.get_text()
        line = re.sub(r'\[\d+\]', '', line)
        line = re.sub(r'\s+', ' ', line)
        line = line.strip()
        if line:
            paragraphs.append(line)

    if not paragraphs:
        continue

    if 'Воспоминания' in heading:
        for i, line in enumerate(paragraphs):
            if line.startswith('Предчувствие меня не обмануло'):
                letter = {
                    'дата': None,
                    'автор': 'Эрдман',
                    'текст': ' '.join(paragraphs[i:])
                }
                rows.append(letter)
            elif line.startswith('«Если мое присутствие') or line.startswith('«Не обвиняй их'):
                letter = {
                    'дата': '1931',
                    'автор': 'Эрдман',
                    'текст': ' '.join(paragraphs[i:i + 2])
                }
                rows.append(letter)
        continue

    if paragraphs[0].startswith('Кажется, моя первая ночь в Томске'):
        current_year = 1935

    header = []
    date = None
    for line in paragraphs:
        date_parts = read_date(line, current_year)
        if date_parts[0]:
            date = date_parts[0]
            current_year = date_parts[1]
        else:
            address = line.rstrip(',.') in address_lines
            for start in address_starts:
                if line.lower().startswith(start.lower()):
                    address = True
                    break
            if not address:
                break
        header.append(line)

    paragraphs = paragraphs[len(header):]

    letter = []
    for line in paragraphs:
        if line.startswith('Они встретились в квартире Бориса Робертовича'):
            break
        letter.append(line)

    if letter:
        date_parts = read_date(letter[-1], current_year)
        last_date = date_parts[0]
        if last_date:
            if date is None:
                date = last_date
            letter = letter[:-1]

    signatures = []
    for line in letter[-3:]:
        signatures.append(line.rstrip('.'))

    text = ' '.join(letter)
    header_text = ' '.join(header).lower()
    if re.search(r'степановой|эрдман', header_text):
        author = 'Эрдман'
    elif 'Лина' in signatures:
        author = 'Степанова'
    elif text.startswith('Вчера имела разговор') or text.startswith('Елка моя сияет'):
        author = 'Степанова'
    elif 'Николай.' in text or 'Линуш' in text or 'Худыра' in text:
        author = 'Эрдман'
    elif 'перетаскивал их' in text or text.startswith('Телеграмма'):
        author = 'Эрдман'
    else:
        author = None

    if letter:
        row = {
            'дата': date,
            'автор': author,
            'текст': text
        }
        rows.append(row)

In [4]:
morph = pymorphy3.MorphAnalyzer()

for row in rows:
    text = row['текст'].lower()
    text = re.sub(r'[^а-яёa-z ]', ' ', text)
    tokens = tokenize(text)

    clean_tokens = []
    for token in tokens:
        word = token.text
        if word.isalnum():
            lemma = morph.parse(word)[0].normal_form
            clean_tokens.append(lemma)

    row['леммы'] = ' '.join(clean_tokens)
    row['количество слов'] = len(clean_tokens)

In [5]:
df = pd.DataFrame(rows)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   дата             62 non-null     str  
 1   автор            77 non-null     str  
 2   текст            77 non-null     str  
 3   леммы            77 non-null     str  
 4   количество слов  77 non-null     int64
dtypes: int64(1), str(4)
memory usage: 3.1 KB


In [6]:
df['автор'].value_counts()

автор
Эрдман       46
Степанова    31
Name: count, dtype: int64

In [7]:
df.head()

,дата,автор,текст,леммы,количество слов
0,NaN,Эрдман,Через несколько дней после вашего отъезда я за...,через несколько день после ваш отъезд я запере...,208
1,NaN,Эрдман,Мне было очень хорошо с Вами. Я никогда не заб...,я быть очень хорошо с вы я никогда не забыть в...,144
2,1933-11-03,Эрдман,Только что задавал корм своему Гнедко (Петушок...,только что задавать корм свой гнедко петушок с...,111
3,1933-11-04,Степанова,"Дорогой, прекрасный, любимый, мне хочется, что...",дорога прекрасный любимый я хотеться чтобы это...,122
4,1933-11-05,Степанова,"Дорогой мой, посылка халтурно составлена, наск...",дорога мой посылка халтурно составить наскоро ...,28
